In [1]:
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.8 MB/s eta 0:00:00


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="5g1wHshIyPMLah4nnkvD")
project = rf.workspace("sania-akter").project("a8-kykao")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to a8-1 in yolov8:: 100%|██████████| 806/806 [00:00<00:00, 10207.92it/s]


In [3]:
import os
import yaml

DATASET_DIR = "/content/a8-1"

print("Dataset exists:", os.path.exists(DATASET_DIR))
print("\nDataset contents:")
print(os.listdir(DATASET_DIR))

with open(f"{DATASET_DIR}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print("\nClasses:")
print(data["names"])

print("\nNumber of classes:")
print(data["nc"])

Dataset exists: True

Dataset contents:
['test', 'README.roboflow.txt', 'data.yaml', 'valid', 'train']

Classes:
['lichi', 'mehegoni', 'pen']

Number of classes:
3


In [4]:
from pathlib import Path

for split in ["train", "valid", "test"]:
    img_dir = Path(DATASET_DIR) / split / "images"
    label_dir = Path(DATASET_DIR) / split / "labels"

    images = list(img_dir.glob("*"))
    labels = list(label_dir.glob("*.txt"))

    print(f"{split}:")
    print("  Images :", len(images))
    print("  Labels :", len(labels))

train:
  Images : 351
  Labels : 351
valid:
  Images : 33
  Labels : 33
test:
  Images : 17
  Labels : 17


In [5]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.5 MB/s eta 0:00:00


In [6]:
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

import torch
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.4.118
GPU available: True
GPU: Tesla T4


In [7]:
from pathlib import Path
import yaml

DATASET_DIR = "/content/a8-1"

with open(f"{DATASET_DIR}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print(data)

{'names': ['lichi', 'mehegoni', 'pen'], 'nc': 3, 'roboflow': {'license': 'Private', 'project': 'a8-kykao', 'url': 'https://app.roboflow.com/sania-akter/a8-kykao/1', 'version': 1, 'workspace': 'sania-akter'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [8]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

In [9]:
results = model.train(
    data="/content/a8-1/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/runs",
    name="yolov8n_custom",
    patience=10,
    verbose=True
)

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/a8-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_custom, nbs=64, nms=False, op

In [10]:
from ultralytics import YOLO

best_model = YOLO(
    "/content/runs/yolov8n_custom/weights/best.pt"
)

test_results = best_model.val(
    data="/content/a8-1/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1408.7±350.7 MB/s, size: 44.8 KB)
val: Scanning /content/a8-1/test/labels... 17 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 17/17 2.0Kit/s 0.0s
val: New cache created: /content/a8-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.4it/s 0.6s
                   all         17         18      0.888      0.833      0.837      0.518
                 lichi          6          6      0.997      0.667       0.83      0.496
              mehegoni          6          6        0.7      0.833      0.686       0.45
                   pen          6          6      0.968          1      0.995      0.608
Speed: 7.7ms preprocess, 13.3ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved 

In [11]:
test_predictions = best_model.predict(
    source="/content/a8-1/test/images",
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/runs",
    name="test_predictions"
)


image 1/17 /content/a8-1/test/images/08ec8155-533e-46bf-be64-d8d67bd23d83_jpg.rf.05af87d362bb2a0ec5d6e7614ae54fd5.jpg: 640x640 2 pens, 7.2ms
image 2/17 /content/a8-1/test/images/0c178b1e-70ac-43d2-8fe1-ae9ddfd91bf7_jpg.rf.7ed0675dea665e77583f7551a711e8bb.jpg: 640x640 1 pen, 7.2ms
image 3/17 /content/a8-1/test/images/1d22c719-13b3-47d3-9fef-3d225559a8f6_jpg.rf.82fe0ecf76521674675f310e134e2108.jpg: 640x640 1 pen, 7.2ms
image 4/17 /content/a8-1/test/images/6bbf30b6-8f53-4155-93b5-a4aa0bba8c3e_jpg.rf.1ff106b5e7480c82d3576d6f25fc71ba.jpg: 640x640 1 pen, 7.2ms
image 5/17 /content/a8-1/test/images/6f854482-3a99-4f5a-8aa9-594f50847abe_jpg.rf.bb66560e29f00282a73a2a80e3093e11.jpg: 640x640 3 pens, 7.2ms
image 6/17 /content/a8-1/test/images/8c37af7d-61a5-4faf-9795-1055f9046f10_jpg.rf.9df4b7ec5e7b41e9d7d4fc7db93cb9df.jpg: 640x640 1 pen, 7.2ms
image 7/17 /content/a8-1/test/images/LichiLeaf_Realme6i_224x224_126_jpg.rf.76b56c7ccb7253c953e9e09c26e26e1f.jpg: 640x640 1 lichi, 7.2ms
image 8/17 /content/a

In [12]:
base_model = YOLO("yolov8n.pt")

In [13]:
base_predictions = base_model.predict(
    source="/content/a8-1/test/images",
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/runs",
    name="pretrained_predictions"
)


image 1/17 /content/a8-1/test/images/08ec8155-533e-46bf-be64-d8d67bd23d83_jpg.rf.05af87d362bb2a0ec5d6e7614ae54fd5.jpg: 640x640 (no detections), 7.7ms
image 2/17 /content/a8-1/test/images/0c178b1e-70ac-43d2-8fe1-ae9ddfd91bf7_jpg.rf.7ed0675dea665e77583f7551a711e8bb.jpg: 640x640 (no detections), 7.7ms
image 3/17 /content/a8-1/test/images/1d22c719-13b3-47d3-9fef-3d225559a8f6_jpg.rf.82fe0ecf76521674675f310e134e2108.jpg: 640x640 1 toothbrush, 7.7ms
image 4/17 /content/a8-1/test/images/6bbf30b6-8f53-4155-93b5-a4aa0bba8c3e_jpg.rf.1ff106b5e7480c82d3576d6f25fc71ba.jpg: 640x640 (no detections), 7.7ms
image 5/17 /content/a8-1/test/images/6f854482-3a99-4f5a-8aa9-594f50847abe_jpg.rf.bb66560e29f00282a73a2a80e3093e11.jpg: 640x640 (no detections), 7.7ms
image 6/17 /content/a8-1/test/images/8c37af7d-61a5-4faf-9795-1055f9046f10_jpg.rf.9df4b7ec5e7b41e9d7d4fc7db93cb9df.jpg: 640x640 2 toothbrushs, 7.6ms
image 7/17 /content/a8-1/test/images/LichiLeaf_Realme6i_224x224_126_jpg.rf.76b56c7ccb7253c953e9e09c26e26

In [14]:
import time
import glob
import numpy as np
from ultralytics import YOLO

model = YOLO(
    "/content/runs/yolov8n_custom/weights/best.pt"
)

test_images = glob.glob("/content/a8-1/test/images/*")

# Warm-up
for img in test_images[:5]:
    model.predict(
        img,
        imgsz=640,
        device=0,
        verbose=False
    )

latencies = []

for img in test_images:
    start = time.perf_counter()

    model.predict(
        img,
        imgsz=640,
        device=0,
        verbose=False
    )

    end = time.perf_counter()

    latencies.append((end - start) * 1000)

avg_latency = np.mean(latencies)
median_latency = np.median(latencies)
fps = 1000 / avg_latency

print(f"Average latency : {avg_latency:.2f} ms/image")
print(f"Median latency  : {median_latency:.2f} ms/image")
print(f"Approx. FPS     : {fps:.2f}")

Average latency : 11.96 ms/image
Median latency  : 11.91 ms/image
Approx. FPS     : 83.63


In [15]:
predictions = model.predict(
    source="/content/a8-1/test/images",
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/runs",
    name="final_test_predictions"
)

Results saved to /content/runs/final_test_predictions
